In [4]:
import pandas as pd

from surprise import Dataset, Reader
from surprise import SVD, NMF
from surprise.model_selection import cross_validate, train_test_split, GridSearchCV

### Load data


In [5]:
df = pd.read_csv("data/ratings_beauty.csv")

reader = Reader(rating_scale=(1, 10))
data = Dataset.load_from_df(df[['UserId', 'ProductId', 'Rating']], reader)


In [6]:
print('Number of ratings: %d\nNumber of ProductIds: %d\nNumber of users: %d' % (len(df), len(df['Rating'].unique()), len(df['UserId'].unique())))

Number of ratings: 2023070
Number of ProductIds: 5
Number of users: 1210271


## Apply SVD

In [7]:
trainset, testset = train_test_split(data, test_size=0.2)

model = SVD(n_factors=80, n_epochs=20, lr_all=0.005, reg_all=0.2)
model.fit(trainset)
predictions = model.test(testset)

In [8]:
df.head(5)

,UserId,ProductId,Rating,Timestamp
0,A39HTATAQ9V7YF,0205616461,5.0,1369699200
1,A3JM6GV9MNOF9X,0558925278,3.0,1355443200
2,A1Z513UWSAAO0F,0558925278,5.0,1404691200
3,A1WMRR494NWEWV,0733001998,4.0,1382572800
4,A3IAAVS479H7M7,0737104473,1.0,1274227200


In [9]:
df[df['UserId'] == 'A39HTATAQ9V7YF']

,UserId,ProductId,Rating,Timestamp
0,A39HTATAQ9V7YF,0205616461,5.0,1369699200
899125,A39HTATAQ9V7YF,B002OVV7F0,3.0,1369699200
969482,A39HTATAQ9V7YF,B0031IH5FQ,5.0,1369699200
1499680,A39HTATAQ9V7YF,B006GQPZ8E,4.0,1369699200


In [10]:
model.predict('A39HTATAQ9V7YF', '0205616461' )

Prediction(uid='A39HTATAQ9V7YF', iid='0205616461', r_ui=None, est=np.float64(4.416949330995847), details={'was_impossible': False})

In [11]:
predictions

[Prediction(uid='A1YF3HLZ5ORJW5', iid='B006QO4BRM', r_ui=4.0, est=np.float64(3.972918296127552), details={'was_impossible': False}),
 Prediction(uid='AB8NIIM9UPMC7', iid='B000IOAAQ2', r_ui=5.0, est=np.float64(4.0259595548640394), details={'was_impossible': False}),
 Prediction(uid='A2I9EYS0ZPPP0J', iid='B001KYRP80', r_ui=5.0, est=np.float64(3.7025652069033415), details={'was_impossible': False}),
 Prediction(uid='A1M9MY9F95L3JI', iid='B004SPDEWE', r_ui=5.0, est=np.float64(4.249346753716983), details={'was_impossible': False}),
 Prediction(uid='A2T80QPEG88K86', iid='B002JCSLXA', r_ui=5.0, est=np.float64(3.9167545876907295), details={'was_impossible': False}),
 Prediction(uid='A2JP02PCZ7YCZE', iid='B008ZFDOMS', r_ui=4.0, est=np.float64(4.215006167255477), details={'was_impossible': False}),
 Prediction(uid='A1XRW9OP3U63V8', iid='B001A1P9QS', r_ui=4.0, est=np.float64(3.6973760832626272), details={'was_impossible': False}),
 Prediction(uid='A37O8LMWANI620', iid='B002HFKPLU', r_ui=5.0, est=

In [12]:
# Perform cross-validation
cv_results = cross_validate(model, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)



Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    1.2472  1.2448  1.2446  1.2459  1.2436  1.2452  0.0012  
MAE (testset)     0.9810  0.9796  0.9799  0.9806  0.9788  0.9800  0.0008  
Fit time          29.72   28.74   29.20   31.43   30.01   29.82   0.92    
Test time         3.68    3.56    3.54    3.67    3.61    3.61    0.05    


In [13]:
def get_recommendations(model, user_id, n_recommendations=5):   
    # Get all product IDs
    all_product_ids = df['ProductId'].unique()
    
    # Get the products the user has already rated
    rated_products = df[df['UserId'] == user_id]['ProductId'].unique()
    
    # Filter out the products the user has already rated
    unrated_products = [pid for pid in all_product_ids if pid not in rated_products]
    
    # Predict ratings for the unrated products
    predictions = [model.predict(user_id, pid) for pid in unrated_products]
    
    # Sort predictions by estimated rating in descending order
    predictions.sort(key=lambda x: x.est, reverse=True)
    
    # Get the top N recommendations
    top_recommendations = predictions[:n_recommendations]
    
    return [(pred.iid, pred.est) for pred in top_recommendations]

In [14]:
get_recommendations(model, 'A39HTATAQ9V7YF', n_recommendations=5)

[('B004AI5756', np.float64(4.871982181775004)),
 ('B001F0RBRE', np.float64(4.8511754679879635)),
 ('B0018QHU6W', np.float64(4.822819207260653)),
 ('B008CYHZS6', np.float64(4.820807796293814)),
 ('B00466VJ04', np.float64(4.820402616843522))]